In [43]:
import os
import re
import sys
from pathlib import Path
from typing import List, Dict
from langchain_core.prompts import ChatPromptTemplate
import re
import numpy as np
import pandas as pd
import requests
import rapidfuzz
import json
from langchain_core.output_parsers import StrOutputParser
from agent.prompts import EXPLAIN_PROMPT
from sentence_transformers import SentenceTransformer
sys.path.append(str(Path.cwd().parent))  # Add parent directory to sys.path for local imports

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from agent.agents.explainer import explain_rule
from agent.tools.mvel_parser_tool import parse_mvel_branches
from agent.llm import get_llm

In [51]:
EXPLAIN_PROMPT = ChatPromptTemplate.from_messages([
        (
            "system",
            "You write for non-technical stakeholders. "
            "Do not mention code, syntax, or programming terms."
        ),
        (
            "user",
            "Convert the following extracted rule structure into clear English.\n\n"
            "Requirements:\n"
            "- Start with a 1–2 sentence Summary\n"
            "- Then list Decision logic as bullet points\n"
            "- One bullet per branch, in order\n"
            "- Use 'Otherwise,' for the DEFAULT branch\n"
            "- Use provided context to define business terms if relevant\n\n"
            "Context:\n"
            "{context}\n\n"     
            "Rule extraction (JSON):\n"
            "{extraction_json}"
        )
    ])

# NO PARSING
EXPLAIN_V2 = ChatPromptTemplate.from_messages([
        (
            "system",
            "You write for non-technical stakeholders. "
            "Do not mention code, syntax, or programming terms."
        ),
        (
            "user",
            "Convert the following MVEL text into clear, natural English.\n"
            "Explain what the expression does in plain language.\n"
            "Break down each part of the expression step by step.\n"
            "If there are conditions, operators, or functions, describe what they mean.\n"
            "Provide the final meaning as if explaining to someone with no programming background.\n"
            "MVEL TEXT:\n"
            "{mvel_text}"
        )
    ])


In [ ]:

    # Ensure llm is initialized and explain_rule is called with (llm, extraction, context)
def explain_rule(rule):
    llm = get_llm(model="llama3.1", temperature=0.0)

    # prepare parsed extraction and context
    parsed = parse_mvel_branches(rule)
    context = "Business context or glossary here"  # set as needed


    # Option B — feed parsed JSON directly into the prompt/chain
    extraction_json = json.dumps(parsed, ensure_ascii=False, indent=2)
    chain = EXPLAIN_PROMPT | llm | StrOutputParser()
    explanation = chain.invoke({"extraction_json": extraction_json, "context": context}).strip()
    return explanation
    
    





In [57]:
import re
import pandas as pd
from rapidfuzz import fuzz

CSV_PATH = "pre_match.csv"

tests = [
    "(input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equals.IgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')",
    "(input.message != null && input.message.equalsIgnoreCase('ABC')) && (input.estimate != null && input.estimate.equalsIgnoreCase('money')) && (input.srcworkType != null && input.srcworkType. equalsIgnoreCase('normal')) && (input.clientId != null && input.clientId != 'Y') && (input.transactionIndex == 'N') && (input.Ref.contains('CASH MONEY HEROES'))",
    "(input.message != null && (input.message.equalsIgnoreCase('FT900') || input.messageTypeCode.equalsIgnoreCase('FT500') || input.message.equalsIgnoreCase('SSN') || input.message.equalsIgnoreCase('MT'))) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transactionCode.equalsIgnoreCase('CAPITAL HILL')) && (input.entityFlag != null && input.entityFlag == 'Y')",
]

def normalize(text: str) -> str:
    t = str(text).strip()
    t = re.sub(r"\s+", " ", t)
    t = t.replace(" | | ", " || ")
    return t

def similarity(a: str, b: str) -> float:
    # 0..1 similarity score
    return fuzz.token_set_ratio(normalize(a), normalize(b)) / 100.0

def best_target_match(model_output: str, library_targets: list[str]) -> tuple[int, float]:
    scores = [similarity(model_output, t) for t in library_targets]
    best_idx = max(range(len(scores)), key=lambda i: scores[i])
    return best_idx, float(scores[best_idx])


    # Requires: pip install rapidfuzz
def evaluate(CSV_PATH, explain_rule: explain_rule):
    df = pd.read_csv(CSV_PATH)

    df = df.dropna(subset=["RULE", "TARGET"]).reset_index(drop=True)

    library_rules = df["RULE"].astype(str).tolist()
    library_targets = df["TARGET"].astype(str).tolist()

    print(f"Loaded library rows: {len(library_rules)}")

    for i, test_rule in enumerate(tests, start=1):
        # You must have explain_rule available in your environment.
        # If your function name is different, replace this call.
        model_output = explain_rule(test_rule)

        best_idx, best_score = best_target_match(model_output, library_targets)

        print("\n" + "=" * 100)
        print(f"TEST {i}")
        print("TEST RULE:\n", test_rule)
        print("\nMODEL OUTPUT:\n", model_output)

        print("\nBEST MATCHED TRAINING TARGET (fuzzy string match):")
        print(f"Index: {best_idx} | score: {best_score:.4f}")
        print("TRAINING TARGET:", library_targets[best_idx])
        print("TRAINING RULE  :", library_rules[best_idx])

evaluate(CSV_PATH=CSV_PATH, explain_rule=explain_rule)


Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equals.IgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')

MODEL OUTPUT:
 Here is the extracted rule structure rewritten in clear English:

**Summary**
This rule checks if a message meets certain conditions and advises on whether to proceed with a transaction.

**Decision Logic**

* The message must be 'MT', 'saipan', or 'CA' (case-insensitive) and not null.
* The money code must be 'Actual' (case-insensitive) and not null.
* The transaction type must be 'CAPITAL HILL' (case-insensitive) and not null.
* The advise flag must be set to 'Y'.

**Note:** If all conditions are met, the rule will proceed with advis